<a href="https://colab.research.google.com/github/devanshubisht461-dev/dmart-regional-director-assistant/blob/main/Retail_Capstone_Part_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Create project folders
folders = [
    "data",
    "scripts",
    "analysis"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [2]:
"""Seeded generator for the three practice datasets. Standard library only.
Run from the repository root:  python scripts/generate_data.py
Writes data/store_daily.csv, data/inventory.csv and data/feedback.csv.
"""
import csv
import os
import random
from datetime import date, timedelta

SEED = 2604
os.makedirs("data", exist_ok=True)
rng = random.Random(SEED)
START = date(2026, 4, 1)
DAYS = 90  # 2026-04-01 .. 2026-06-29

# store_id, store, base_footfall, conversion, basket_inr, daily_target_inr, marketing_base_inr, stockout_mean_hours
STORES = [
    ("S01", "Mumbai",    2300, 0.52, 880, 1050000, 52000, 2.0),
    ("S02", "Pune",      1500, 0.50, 860,  900000, 41000, 2.6),
    ("S03", "Hyderabad", 2150, 0.51, 840,  950000, 47000, 3.9),
    ("S04", "Bengaluru", 2050, 0.53, 870,  950000, 49000, 2.4),
    ("S05", "Ahmedabad", 1900, 0.52, 820,  850000, 36000, 3.1),
]

# ---------- store_daily ----------
daily = []
for sid, name, base_ff, conv, basket, target, mkt_base, so_mean in STORES:
    for d in range(DAYS):
        day = START + timedelta(days=d)
        weekend = day.weekday() >= 5
        marketing = mkt_base * rng.gauss(1.0, 0.15)
        footfall = base_ff * (1.05 if weekend else 1.0) * rng.uniform(0.88, 1.12)
        footfall *= 1 + 0.15 * (marketing / mkt_base - 1)
        stockout = max(0.0, round(rng.gauss(so_mean, 1.2), 1))
        discount = round(rng.uniform(3.0, 12.0), 1)
        revenue = footfall * rng.gauss(conv, 0.02) * rng.gauss(basket, 25) * (1 - 0.012 * stockout)
        daily.append({
            "date": day.isoformat(), "store_id": sid, "store": name,
            "footfall": int(round(footfall)), "revenue_inr": round(revenue, 2),
            "daily_target_inr": target, "marketing_spend_inr": int(round(marketing)),
            "stockout_hours": stockout, "discount_pct": discount,
        })

def row(sid, day_offset):
    return next(r for r in daily
                if r["store_id"] == sid and r["date"] == (START + timedelta(days=day_offset)).isoformat())

# data-quality issues injected on purpose
for sid, off in [("S02", 11), ("S02", 46), ("S01", 29), ("S04", 70)]:
    row(sid, off)["footfall"] = ""              # 4 missing footfall values
row("S03", 20)["footfall"] = -150               # impossible negative footfall
row("S05", 55)["footfall"] = 9800               # extreme high footfall
row("S01", 8)["footfall"] = 6100                # extreme high footfall
for sid, off in [("S01", 40), ("S03", 60), ("S05", 15)]:
    daily.append(dict(row(sid, off)))           # 3 exact duplicate rows appended at the end

with open("data/store_daily.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(daily[0].keys()))
    w.writeheader()
    w.writerows(daily)

# ---------- inventory ----------
PRODUCTS = [("P01", "Amul Milk 1L"), ("P02", "Toor Dal 1kg"), ("P03", "Basmati Rice 5kg"),
            ("P04", "Sunflower Oil 1L"), ("P05", "Iodised Salt 1kg"), ("P06", "Instant Noodles 70g"),
            ("P07", "Glucose Biscuits 200g"), ("P08", "Detergent Powder 1kg")]
inventory = []
for sid, name, *_ in STORES:
    for pid, pname in PRODUCTS:
        threshold = rng.randint(60, 150)
        stock = rng.randint(20, 400)
        inventory.append({"store_id": sid, "store": name, "product_id": pid, "product": pname,
                          "stock_units": stock, "reorder_threshold": threshold})
for r in inventory:                              # make the Amul Milk answer deliberate
    if r["product_id"] == "P01":
        r["reorder_threshold"] = 100
        r["stock_units"] = {"Mumbai": 240, "Pune": 85, "Hyderabad": 40, "Bengaluru": 130, "Ahmedabad": 100}[r["store"]]
with open("data/inventory.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(inventory[0].keys()))
    w.writeheader()
    w.writerows(inventory)

# ---------- feedback ----------
THEMES = ["Stock availability", "Queue time", "Pricing", "Freshness", "Staff behaviour"]
THEME_W = {"Mumbai": [1, 4, 2, 2, 1], "Pune": [1, 2, 5, 1, 1], "Hyderabad": [6, 2, 1, 1, 1],
           "Bengaluru": [2, 2, 2, 3, 1], "Ahmedabad": [2, 3, 1, 2, 2]}
RATING_W = {"Mumbai": [1, 1, 3, 5, 4], "Pune": [1, 2, 3, 4, 3], "Hyderabad": [3, 3, 3, 3, 2],
            "Bengaluru": [1, 1, 3, 5, 4], "Ahmedabad": [1, 2, 4, 4, 3]}
feedback = []
n = 0
for sid, name, *_ in STORES:
    for _ in range(40):
        n += 1
        day = START + timedelta(days=rng.randint(0, DAYS - 1))
        rating = rng.choices([1, 2, 3, 4, 5], weights=RATING_W[name], k=1)[0]
        theme = rng.choices(THEMES, weights=THEME_W[name], k=1)[0]
        feedback.append({"feedback_id": f"F{n:03d}", "store_id": sid, "store": name,
                         "date": day.isoformat(), "rating": rating, "theme": theme})
for i in [7, 33, 58, 71, 90, 104, 120, 133, 151, 166, 178, 195]:   # 12 unrated rows
    feedback[i]["rating"] = ""
with open("data/feedback.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(feedback[0].keys()))
    w.writeheader()
    w.writerows(feedback)
print(len(daily), len(inventory), len(feedback))


453 40 200


In [3]:
import os
import pandas as pd

files = [
    "data/store_daily.csv",
    "data/inventory.csv",
    "data/feedback.csv"
]

for file in files:
    df = pd.read_csv(file)
    print("\nFile:", file)
    print("Rows and columns:", df.shape)
    print(df.head(3))


File: data/store_daily.csv
Rows and columns: (453, 9)
         date store_id   store  footfall  revenue_inr  daily_target_inr  \
0  2026-04-01      S01  Mumbai    2334.0   1062581.15           1050000   
1  2026-04-02      S01  Mumbai    2178.0    949323.45           1050000   
2  2026-04-03      S01  Mumbai    2351.0   1034399.82           1050000   

   marketing_spend_inr  stockout_hours  discount_pct  
0                72267             2.1           4.1  
1                45653             0.8           9.5  
2                52957             2.2           8.3  

File: data/inventory.csv
Rows and columns: (40, 6)
  store_id   store product_id           product  stock_units  \
0      S01  Mumbai        P01      Amul Milk 1L          240   
1      S01  Mumbai        P02      Toor Dal 1kg          209   
2      S01  Mumbai        P03  Basmati Rice 5kg          252   

   reorder_threshold  
0                100  
1                136  
2                 78  

File: data/feedback.cs

In [4]:
import pandas as pd
import numpy as np

# Load all three datasets
daily = pd.read_csv("data/store_daily.csv")
inventory = pd.read_csv("data/inventory.csv")
feedback = pd.read_csv("data/feedback.csv")

# Inspect datasets
datasets = {
    "Store Daily": daily,
    "Inventory": inventory,
    "Feedback": feedback
}

for name, df in datasets.items():

    print("\n" + "="*50)
    print("DATASET:", name)
    print("="*50)

    print("\nShape:")
    print(df.shape)

    print("\nColumn Information:")
    print(df.info())

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nStatistical Summary:")
    print(df.describe(include="all"))


DATASET: Store Daily

Shape:
(453, 9)

Column Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 453 entries, 0 to 452
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 453 non-null    object 
 1   store_id             453 non-null    object 
 2   store                453 non-null    object 
 3   footfall             449 non-null    float64
 4   revenue_inr          453 non-null    float64
 5   daily_target_inr     453 non-null    int64  
 6   marketing_spend_inr  453 non-null    int64  
 7   stockout_hours       453 non-null    float64
 8   discount_pct         453 non-null    float64
dtypes: float64(4), int64(2), object(3)
memory usage: 32.0+ KB
None

Missing Values:
date                   0
store_id               0
store                  0
footfall               4
revenue_inr            0
daily_target_inr       0
marketing_spend_inr    0
stockout_hours         0
d

In [5]:
import os

os.makedirs("analysis", exist_ok=True)

report = """
# DATA QUALITY REPORT
## Retail Stockout Analysis – Pune

### 1. Dataset Overview

Three datasets were generated using the provided
seeded Python generator.

| Dataset | Records | Columns |
|---|---:|---:|
| Store Daily | 453 | 9 |
| Inventory | 40 | 6 |
| Feedback | 200 | 6 |

### 2. Store Daily Data Quality Issues

- Four missing footfall values.
- One negative footfall value (-150).
- Two extreme footfall values (6100 and 9800).
- Three exact duplicate records.

### 3. Inventory Data Quality

- 40 records inspected.
- No missing values.
- No duplicate records.
- No negative stock quantities.

### 4. Feedback Data Quality

- 200 records inspected.
- 12 missing customer ratings.
- No duplicate records.
- Ratings range from 1 to 5.

### 5. Cleaning Recommendations

1. Remove exact duplicate daily records.
2. Treat negative footfall as invalid.
3. Investigate extreme footfall observations.
4. Impute missing footfall using an appropriate
   store-level median.
5. Preserve missing feedback ratings as missing.

### 6. Conclusion

The datasets contain identifiable quality issues
that must be addressed before performing reliable
business analysis.
"""

with open("analysis/data_quality_report.md", "w") as f:
    f.write(report)

print("Data Quality Report created successfully!")

Data Quality Report created successfully!


In [6]:
import pandas as pd
import numpy as np
import os

# Load original raw dataset
daily = pd.read_csv("data/store_daily.csv")

# Keep track of cleaning operations
cleaning_log = []

original_rows = len(daily)

# STEP 1: Remove exact duplicates
duplicates = daily.duplicated().sum()

daily = daily.drop_duplicates().copy()

cleaning_log.append(
    f"Removed {duplicates} exact duplicate rows."
)

# STEP 2: Convert negative footfall into missing values
negative_count = (daily["footfall"] < 0).sum()

daily.loc[daily["footfall"] < 0, "footfall"] = np.nan

cleaning_log.append(
    f"Converted {negative_count} negative footfall value(s) to missing."
)

# STEP 3: Identify and cap extreme footfall using store-wise IQR
outlier_count = 0

for store in daily["store"].unique():

    mask = daily["store"] == store

    values = daily.loc[mask, "footfall"].dropna()

    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = mask & (
        (daily["footfall"] < lower_limit) |
        (daily["footfall"] > upper_limit)
    )

    outlier_count += outliers.sum()

    daily.loc[outliers, "footfall"] = daily.loc[
        outliers, "footfall"
    ].clip(lower=lower_limit, upper=upper_limit)

cleaning_log.append(
    f"Capped {outlier_count} extreme footfall value(s) using store-wise IQR limits."
)

# STEP 4: Impute missing footfall using store-wise median
missing_before = daily["footfall"].isna().sum()

daily["footfall"] = daily.groupby("store")["footfall"].transform(
    lambda x: x.fillna(x.median())
)

cleaning_log.append(
    f"Imputed {missing_before} missing footfall value(s) using store-wise median."
)

# STEP 5: Preserve missing feedback ratings
feedback = pd.read_csv("data/feedback.csv")

missing_ratings = feedback["rating"].isna().sum()

cleaning_log.append(
    f"Preserved {missing_ratings} missing feedback ratings without imputation."
)

# Save cleaned dataset
os.makedirs("data", exist_ok=True)

daily.to_csv("data/store_daily_clean.csv", index=False)

# Save cleaning log
os.makedirs("analysis", exist_ok=True)

with open("analysis/clean_log.md", "w") as f:

    f.write("# DATA CLEANING LOG\n\n")

    for i, item in enumerate(cleaning_log, 1):

        f.write(f"{i}. {item}\n\n")

print("DATA CLEANING COMPLETED")
print("-----------------------")

print("Original rows:", original_rows)
print("Cleaned rows:", len(daily))
print("Duplicates removed:", duplicates)
print("Negative values corrected:", negative_count)
print("Extreme values capped:", outlier_count)
print("Missing footfall imputed:", missing_before)
print("Missing feedback ratings preserved:", missing_ratings)

print("\nRemaining missing footfall:", daily["footfall"].isna().sum())
print("Remaining duplicates:", daily.duplicated().sum())

print("\nCleaned dataset saved successfully!")


DATA CLEANING COMPLETED
-----------------------
Original rows: 453
Cleaned rows: 450
Duplicates removed: 3
Negative values corrected: 1
Extreme values capped: 2
Missing footfall imputed: 5
Missing feedback ratings preserved: 12

Remaining missing footfall: 0
Remaining duplicates: 0

Cleaned dataset saved successfully!


In [7]:
import pandas as pd
import numpy as np

# Load cleaned dataset
daily = pd.read_csv("data/store_daily_clean.csv")

# 1. Revenue per footfall
daily["revenue_per_footfall"] = (
    daily["revenue_inr"] / daily["footfall"]
)

# 2. Achievement percentage
daily["achievement_pct"] = (
    daily["revenue_inr"] / daily["daily_target_inr"]
) * 100

# 3. Performance band
daily["performance_band"] = np.where(
    daily["achievement_pct"] >= 100,
    "Target Achieved",
    "Below Target"
)

# 4. Poor performance day
daily["is_poor_day"] = (
    daily["achievement_pct"] < 80
)

# 5. Stockout band
daily["stockout_band"] = pd.cut(
    daily["stockout_hours"],
    bins=[-0.01, 2, 4, float("inf")],
    labels=["Low", "Medium", "High"]
)

# Save updated dataset
daily.to_csv("data/store_daily_clean.csv", index=False)

# Display results
print("FEATURE ENGINEERING COMPLETED")

print("\nNew columns:")
print([
    "revenue_per_footfall",
    "achievement_pct",
    "performance_band",
    "is_poor_day",
    "stockout_band"
])

print("\nSample data:")
print(daily.head())

print("\nPerformance distribution:")
print(daily["performance_band"].value_counts())

print("\nPoor performance days:")
print(daily["is_poor_day"].sum())

FEATURE ENGINEERING COMPLETED

New columns:
['revenue_per_footfall', 'achievement_pct', 'performance_band', 'is_poor_day', 'stockout_band']

Sample data:
         date store_id   store  footfall  revenue_inr  daily_target_inr  \
0  2026-04-01      S01  Mumbai    2334.0   1062581.15           1050000   
1  2026-04-02      S01  Mumbai    2178.0    949323.45           1050000   
2  2026-04-03      S01  Mumbai    2351.0   1034399.82           1050000   
3  2026-04-04      S01  Mumbai    2621.0   1137099.33           1050000   
4  2026-04-05      S01  Mumbai    2431.0   1150075.27           1050000   

   marketing_spend_inr  stockout_hours  discount_pct  revenue_per_footfall  \
0                72267             2.1           4.1            455.261847   
1                45653             0.8           9.5            435.869353   
2                52957             2.2           8.3            439.982909   
3                70594             1.7           6.1            433.841789   
4    

In [8]:
cleaning_decisions = """
# CLEANING DECISIONS AND METHODOLOGY

## 1. Duplicate Records
Three exact duplicate records were removed.
Duplicates can inflate revenue, footfall and other
business performance metrics.

## 2. Negative Footfall
The negative footfall value (-150) was invalid.
It was converted to missing rather than retained.

## 3. Extreme Footfall
Two unusually high values were identified.
Store-wise IQR limits were used to cap these values.
This preserves the observations while reducing
their influence on statistical analysis.

## 4. Missing Footfall
Five missing values were imputed using the median
footfall of the corresponding store.

Median was selected because it is less sensitive
to extreme observations than the mean.

## 5. Missing Customer Ratings
Twelve missing ratings were preserved as missing.
An absent rating does not represent an average rating.

## 6. Original Data Preservation
The original raw datasets were not overwritten.
All cleaning operations were performed on a copy.

## 7. Feature Engineering
The following metrics were created:
- Revenue per footfall
- Achievement percentage
- Performance band
- Poor performance day indicator
- Stockout band

## 8. Analytical Limitation
Correlation does not establish causation.
Further investigation is required before concluding
that stockouts cause revenue decline.
"""

with open("analysis/cleaning_decisions.md", "w") as f:
    f.write(cleaning_decisions)

print("Cleaning methodology documented successfully!")

Cleaning methodology documented successfully!


In [9]:
import pandas as pd
import numpy as np

# Load cleaned and raw datasets
daily = pd.read_csv("data/store_daily_clean.csv")
raw = pd.read_csv("data/store_daily.csv")

# Store performance comparison
store_summary = daily.groupby(
    ["store_id", "store"]
).agg(
    avg_daily_revenue=("revenue_inr", "mean"),
    avg_daily_footfall=("footfall", "mean"),
    avg_stockout_hours=("stockout_hours", "mean"),
    avg_revenue_per_footfall=("revenue_per_footfall", "mean"),
    avg_achievement_pct=("achievement_pct", "mean"),
    poor_days=("is_poor_day", "sum")
).reset_index()

store_summary = store_summary.round(2)

print("STORE PERFORMANCE COMPARISON")
display(store_summary)

# Save results
store_summary.to_csv(
    "analysis/store_performance_comparison.csv",
    index=False
)

STORE PERFORMANCE COMPARISON


,store_id,store,avg_daily_revenue,avg_daily_footfall,avg_stockout_hours,avg_revenue_per_footfall,avg_achievement_pct,poor_days
0,S01,Mumbai,1064371.60,2391.27,1.93,446.05,101.37,1
1,S02,Pune,636358.55,1517.71,2.32,419.57,70.71,85
2,S03,Hyderabad,893233.01,2209.51,3.85,404.60,94.02,4
3,S04,Bengaluru,931065.68,2076.86,2.38,448.23,98.01,0
4,S05,Ahmedabad,782647.37,1922.59,3.19,407.23,92.08,8


In [10]:
# Rank stores by average stockout hours
stockout_ranking = daily.groupby("store").agg(
    avg_stockout_hours=("stockout_hours", "mean")
).sort_values(
    "avg_stockout_hours",
    ascending=False
).reset_index()

stockout_ranking["rank"] = range(
    1, len(stockout_ranking) + 1
)

print("STOCKOUT RANKING")
display(stockout_ranking.round(2))

pune_rank = stockout_ranking[
    stockout_ranking["store"] == "Pune"
]["rank"].iloc[0]

print("Pune's stockout rank:", pune_rank)

if pune_rank == 1:
    print("Pune has the highest average stockout hours.")
else:
    print("Pune does NOT have the highest average stockout hours.")

STOCKOUT RANKING


,store,avg_stockout_hours,rank
0,Hyderabad,3.85,1
1,Ahmedabad,3.19,2
2,Bengaluru,2.38,3
3,Pune,2.32,4
4,Mumbai,1.93,5


Pune's stockout rank: 4
Pune does NOT have the highest average stockout hours.


In [11]:
# Correlation function
def calculate_correlations(df, label):

    print("\n", label)

    print(
        "Revenue vs Stockout:",
        round(
            df["revenue_inr"].corr(df["stockout_hours"]),
            3
        )
    )

    print(
        "Revenue vs Footfall:",
        round(
            df["revenue_inr"].corr(df["footfall"]),
            3
        )
    )


# All stores after cleaning
calculate_correlations(
    daily,
    "ALL STORES - AFTER CLEANING"
)

# Pune after cleaning
pune = daily[daily["store"] == "Pune"]

calculate_correlations(
    pune,
    "PUNE - AFTER CLEANING"
)

# Before-cleaning comparison
calculate_correlations(
    raw,
    "ALL STORES - BEFORE CLEANING"
)

raw_pune = raw[raw["store"] == "Pune"]

calculate_correlations(
    raw_pune,
    "PUNE - BEFORE CLEANING"
)


 ALL STORES - AFTER CLEANING
Revenue vs Stockout: -0.152
Revenue vs Footfall: 0.918

 PUNE - AFTER CLEANING
Revenue vs Stockout: -0.254
Revenue vs Footfall: 0.735

 ALL STORES - BEFORE CLEANING
Revenue vs Stockout: -0.156
Revenue vs Footfall: 0.548

 PUNE - BEFORE CLEANING
Revenue vs Stockout: -0.254
Revenue vs Footfall: 0.744


In [12]:
import os

os.makedirs("analysis", exist_ok=True)

with open("analysis/findings.md", "w") as f:

    f.write("# RETAIL STOCKOUT ANALYSIS – FINDINGS\n\n")

    f.write("## Store Performance Comparison\n\n")
    f.write(store_summary.to_markdown(index=False))
    f.write("\n\n")

    f.write("## Stockout Ranking\n\n")
    f.write(stockout_ranking.round(2).to_markdown(index=False))
    f.write("\n\n")

    f.write("Analysis period: 1 April – 29 June 2026.\n")

print("Initial findings report saved successfully!")

Initial findings report saved successfully!
